## Basic Feature engineering

In [2]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from config.paths import CLEANED_HOTEL_FILE

# Load Data
df = pd.read_csv(CLEANED_HOTEL_FILE)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_nights
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,14,0,Transient,75.0,0,0,Check-Out,2015-07-02,1
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304,0,Transient,75.0,0,0,Check-Out,2015-07-02,1
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240,0,Transient,98.0,0,1,Check-Out,2015-07-03,2
3,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240,0,Transient,98.0,0,1,Check-Out,2015-07-03,2
4,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,No Deposit,14,0,Transient,107.0,0,0,Check-Out,2015-07-03,2


## 1. Temporal Features
- capture seasonality, weekday/weekend effects, lead time categories, and peak season.
| Feature | Description |
|---------|-------------|
| arrival_month_num | Arrival month |
| arrival_week_of_year | Week of arrival |
| arrival_day_of_week | Day of week of arrival |
| lead_time_category | Lead time category (last_minute, short, medium, long) |
| is_weekend_arrival | Weekend arrival indicator |
| arrival_month_sin / arrival_month_cos | Cyclical encoding for month |
| arrival_dow_sin / arrival_dow_cos | Cyclical encoding for day of week |
| is_peak_season | Peak season indicator (Jun-Aug, Dec-Jan) |


In [3]:
df['arrival_date'] = pd.to_datetime(df['reservation_status_date'])
df['arrival_month_num'] = df['arrival_date_month'].map({
    'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
    'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12
})
df['arrival_week_of_year'] = df['arrival_date'].dt.isocalendar().week
df['arrival_day_of_week'] = df['arrival_date'].dt.dayofweek
df['is_weekend_arrival'] = df['arrival_day_of_week'].isin([5,6]).astype(int)
df['lead_time_category'] = pd.cut(df['lead_time'], bins=[-1,7,30,90,500], labels=['last_minute','short','medium','long'])
# Cyclical encoding
df['arrival_month_sin'] = np.sin(2 * np.pi * df['arrival_month_num']/12)
df['arrival_month_cos'] = np.cos(2 * np.pi * df['arrival_month_num']/12)
df['arrival_dow_sin'] = np.sin(2 * np.pi * df['arrival_day_of_week']/7)
df['arrival_dow_cos'] = np.cos(2 * np.pi * df['arrival_day_of_week']/7)
df['is_peak_season'] = df['arrival_month_num'].isin([6,7,8,12,1]).astype(int)

df.shape

(113370, 43)

## 2. Demand Indicators
- measure total guests, total nights, room nights, and market demand index to reflect booking trends.
| Feature | Description |
|---------|-------------|
| total_guests | Total guests (adults + children + babies) |
| total_nights | Total nights stayed |
| room_nights | total_guests × total_nights → Revenue opportunity |
| market_demand_index | Rolling average of total guests per week → Market demand |


In [4]:
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['room_nights'] = df['total_guests'] * df['total_nights']
df['market_demand_index'] = df['total_guests'].rolling(7, min_periods=1).mean()
df.shape


(113370, 46)

## 3. Customer Segmentation
- classify customers into families, groups, business travelers, and estimate loyalty/CLV proxy.
| Feature | Description |
|---------|-------------|
| is_family | Family booking (children>0 or babies>0) |
| is_group | Group of adults (>2 adults) |
| is_business | Business / Corporate customer |
| loyalty_score | Proxy for customer loyalty / CLV |

In [5]:
df['is_family'] = ((df['children']>0) | (df['babies']>0)).astype(int)
df['is_group'] = (df['adults']>2).astype(int)
df['is_business'] = (df['customer_type']=='Contract').astype(int)
df['loyalty_score'] = df['is_repeated_guest'] + df['previous_bookings_not_canceled']
df.shape


(113370, 50)

## 4. Pricing Strategy
- encode deposit, meal packages, and special requests to estimate willingness-to-pay.
| Feature | Description |
|---------|-------------|
| deposit_type_numeric, is_refundable | Deposit type and refundable |
| meal_value_score, has_meal_package | Meal package score |
| has_special_requests, request_intensity | Number and intensity of special requests → willingness-to-pay proxy |


In [6]:
df['deposit_type_numeric'] = df['deposit_type'].map({'No Deposit':0,'Non Refund':0,'Refundable':1})
df['is_refundable'] = (df['deposit_type']=='Refundable').astype(int)
df['meal_value_score'] = df['meal'].map({'SC':0,'BB':1,'HB':2,'FB':3})
df['has_meal_package'] = (df['meal']!='SC').astype(int)
df['has_special_requests'] = (df['total_of_special_requests']>0).astype(int)
df['request_intensity'] = df['total_of_special_requests']/df['total_nights'].replace(0,1)
df.shape


(113370, 56)

## 5. Cancellation Risk
- estimate risk using past cancellations, booking changes, lead time indicators.
| Feature | Description |
|---------|-------------|
| cancellation_risk_score | previous_cancellations + booking_changes |
| last_minute_booking | lead_time < 7 days |
| advance_booking | lead_time > 90 days |


In [7]:
df['cancellation_risk_score'] = df['previous_cancellations'] + df['booking_changes']
df['last_minute_booking'] = (df['lead_time']<7).astype(int)
df['advance_booking'] = (df['lead_time']>90).astype(int)
df.shape


(113370, 59)

## 6. Competitive Intelligence (Synthetic) 5 competitors
- synthetic competitor prices, price gap, percentile, and market volatility to assess competitiveness.
| Feature                                 | Description                                         |
| --------------------------------------- | --------------------------------------------------- |
| competitor_1_price … competitor_5_price | Synthetic competitor prices ±10% of our ADR         |
| competitor_avg_price                    | Average price of the 5 competitors                  |
| competitor_min_price                    | Minimum price among the 5 competitors               |
| competitor_max_price                    | Maximum price among the 5 competitors               |
| price_percentile                        | Percentile rank of our ADR in the market (0-100)    |
| price_competitiveness                   | Our ADR divided by competitor_avg_price             |
| price_gap                               | Difference between our ADR and competitor_avg_price |
| price_trend                             | 7-day rolling mean of ADR                           |
| market_volatility                       | Standard deviation of the 5 competitors’ prices     |



In [8]:
np.random.seed(42)
num_competitors = 5
competitor_prices = []

for i in range(num_competitors):
    competitor_prices.append(df['adr'] * (1 + np.random.uniform(-0.15, 0.15, size=len(df))))

competitor_df = pd.concat(competitor_prices, axis=1)
competitor_df.columns = [f'competitor_{i+1}_price' for i in range(num_competitors)]
df = pd.concat([df, competitor_df], axis=1)

df['competitor_avg_price'] = competitor_df.mean(axis=1)
df['competitor_min_price'] = competitor_df.min(axis=1)
df['competitor_max_price'] = competitor_df.max(axis=1)
df['price_percentile'] = df['adr'].rank(pct=True)
df['price_competitiveness'] = df['adr'] / df['competitor_avg_price']
df['price_gap'] = df['adr'] - df['competitor_avg_price']
df['price_trend'] = df['adr'].rolling(7, min_periods=1).mean()
df['market_volatility'] = competitor_df.std(axis=1)
df.shape


(113370, 72)

## 7. Room & Hotel Features
- room upgrade, hotel type, premium index to capture intrinsic value.
| Feature | Description |
|---------|-------------|
| is_room_upgraded | Whether the room was upgraded |
| hotel_type_encoded | City vs Resort Hotel |
| hotel_premium_index | Premium index = hotel type + peak season |

In [9]:
df['is_room_upgraded'] = (df['reserved_room_type']!=df['assigned_room_type']).astype(int)
df['hotel_type_encoded'] = df['hotel'].map({'City Hotel':0,'Resort Hotel':1})
df['hotel_premium_index'] = df['hotel_type_encoded'] + df['is_peak_season']
df.shape


(113370, 75)

## 8. Interaction Features
- interactions between features like lead_time × total_nights, weekend × family, revenue potential.
| Feature | Description |
|---------|-------------|
| lead_time_x_total_nights | lead_time × total_nights → capture complex patterns |
| is_weekend_x_is_family | Weekend × family |
| deposit_x_cancellation_risk | deposit × cancellation risk |
| adults_per_night | Adults per night |
| revenue_potential | total_guests × total_nights × market_demand_index |
| season_x_market_demand | season factor (-1 low, 0 medium, 1 high) × market_demand_index → captures seasonal demand effects |


In [10]:
df['lead_time_x_total_nights'] = df['lead_time'] * df['total_nights']
df['is_weekend_x_is_family'] = df['is_weekend_arrival'] * df['is_family']
df['deposit_x_cancellation_risk'] = df['deposit_type_numeric'] * df['cancellation_risk_score']
df['adults_per_night'] = df['adults'] / df['total_nights'].replace(0,1)
df['revenue_potential'] = df['total_guests'] * df['total_nights'] * df['market_demand_index']
df['season_factor'] = df['arrival_month_num'].map({
    1:-1, 2:0, 3:0, 4:0, 5:0, 6:1, 7:1, 8:1, 9:0, 10:0, 11:0, 12:1
})
df['season_x_market_demand'] = df['season_factor'] * df['market_demand_index']

df.shape

(113370, 82)

## 9. Lag / Rolling Features (Time Series)
- historical ADR, rolling mean, bookings last 7 days, and booking velocity for time-series effects.
| Feature | Description |
|---------|-------------|
| adr_lag_7d | ADR lag 7 days |
| adr_lag__30d | ADR lag 30 days |
| adr_rolling_mean_7d | 7-day moving average ADR |
| adr_rolling_std_7d | 7-day standard deviation ADR |
| bookings_last_7d | Total guests in the last 7 days |
| booking_velocity | Booking rate per day |

In [11]:
df = df.sort_values('arrival_date')

df['adr_lag_7d'] = df['adr'].shift(7).bfill()     # ใช้ .bfill() แทน fillna(method='bfill')
df['adr_lag_30d'] = df['adr'].shift(30).bfill()   # เช่นเดียวกัน
df['adr_rolling_mean_7d'] = df['adr'].rolling(7, min_periods=1).mean()
df['adr_rolling_std_7d'] = df['adr'].rolling(7, min_periods=1).std().fillna(0)
df['bookings_last_7d'] = df['total_guests'].rolling(7, min_periods=1).sum()
df['booking_velocity'] = df['bookings_last_7d']/7
df.shape

(113370, 88)

In [12]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,adults_per_night,revenue_potential,season_factor,season_x_market_demand,adr_lag_7d,adr_lag_30d,adr_rolling_mean_7d,adr_rolling_std_7d,bookings_last_7d,booking_velocity
69761,City Hotel,1,321,2015,September,36,3,0,2,2,...,1.0,8.0,0,0.0,62.8,62.8,62.8,0.0,2.0,0.285714
69674,City Hotel,1,286,2015,July,31,30,0,2,2,...,1.0,8.0,1,2.0,62.8,62.8,62.8,0.0,4.0,0.571429
69673,City Hotel,1,286,2015,July,31,30,0,2,2,...,1.0,8.0,1,2.0,62.8,62.8,62.8,0.0,6.0,0.857143
69672,City Hotel,1,286,2015,July,31,30,0,2,2,...,1.0,8.0,1,2.0,62.8,62.8,62.8,0.0,8.0,1.142857
69671,City Hotel,1,286,2015,July,31,30,0,2,2,...,1.0,8.0,1,2.0,62.8,62.8,62.8,0.0,10.0,1.428571


In [13]:
df.describe().T


,count,mean,min,25%,50%,75%,max,std
is_canceled,113370.0,0.374482,0.0,0.0,0.0,1.0,1.0,0.483991
lead_time,113370.0,106.113363,0.0,19.0,71.0,164.0,709.0,107.775639
arrival_date_year,113370.0,2016.148179,2015.0,2016.0,2016.0,2017.0,2017.0,0.706403
arrival_date_week_number,113370.0,27.016433,1.0,16.0,27.0,38.0,53.0,13.719448
arrival_date_day_of_month,113370.0,15.772506,1.0,8.0,16.0,23.0,31.0,8.786167
...,...,...,...,...,...,...,...,...
adr_lag_30d,113370.0,98.741297,0.5,70.0,93.6,122.4,211.03,38.880358
adr_rolling_mean_7d,113370.0,98.755846,12.571429,79.193214,97.931429,117.052143,200.428571,26.286533
adr_rolling_std_7d,113370.0,27.55522,0.0,18.080193,27.533405,37.228248,92.337159,14.089236
bookings_last_7d,113370.0,13.586231,2.0,12.0,14.0,15.0,25.0,2.127919


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 113370 entries, 69761 to 36787
Data columns (total 88 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           113370 non-null  object        
 1   is_canceled                     113370 non-null  int64         
 2   lead_time                       113370 non-null  int64         
 3   arrival_date_year               113370 non-null  int64         
 4   arrival_date_month              113370 non-null  object        
 5   arrival_date_week_number        113370 non-null  int64         
 6   arrival_date_day_of_month       113370 non-null  int64         
 7   stays_in_weekend_nights         113370 non-null  int64         
 8   stays_in_week_nights            113370 non-null  int64         
 9   adults                          113370 non-null  int64         
 10  children                        113370 non-null  int64    